# Lab: Decision Tree và Random Forest

## 1. Decision Tree là gì?

Decision Tree (cây quyết định) là một mô hình phân loại/hồi quy dạng *cây nhị phân* (hoặc đa nhánh). Ở mỗi nút trong, mô hình hỏi một câu *về* dữ liệu (kiểu "tuổi > 30 không?"); rẽ trái nếu đúng, rẽ phải nếu sai. Cứ thế cho đến khi đến lá — lá chứa nhãn dự đoán.

Ưu điểm cực lớn: **dễ giải thích**. In cây ra, người không biết Machine Learning vẫn đọc được. Đây là điều mà neural network không làm được.

## 2. Cây học bằng cách nào?

Quy trình chia (split) một nút:
1. Thử mọi feature, mọi ngưỡng có thể.
2. Với mỗi cách chia, đo "chất lượng" bằng một tiêu chí (impurity).
3. Chọn cách chia tốt nhất.
4. Lặp lại đệ quy cho hai nhánh con.
5. Dừng khi: nút thuần (chỉ còn 1 lớp), hoặc đạt `max_depth`, hoặc số mẫu < `min_samples_split`.

## 3. Hai tiêu chí impurity phổ biến

### 3.1. Entropy (information gain — thuật toán ID3/C4.5)

Đo độ "hỗn loạn" của một tập:
$$
H(S) = -\sum_{c=1}^{C} p_c \log_2 p_c
$$
với $p_c$ là tỷ lệ lớp $c$ trong $S$.

**Information Gain** = entropy giảm sau khi chia:
$$
IG(S, \text{split}) = H(S) - \sum_{i} \frac{|S_i|}{|S|} H(S_i)
$$

### 3.2. Gini index (thuật toán CART — sklearn dùng cái này mặc định)
$$
G(S) = 1 - \sum_{c=1}^{C} p_c^2
$$

Cả hai đều đạt cực tiểu (= 0) khi nút thuần (chỉ 1 lớp), cực đại khi các lớp đều nhau. Trong thực tế cho kết quả gần nhau — Gini tính nhanh hơn (không có log).

## 4. Vấn đề lớn: Overfitting

Cây *không bị giới hạn* sẽ học tới khi mỗi lá chỉ có 1 mẫu — train accuracy = 100%, test accuracy thì kém. Cách kiểm soát:

- `max_depth`: giới hạn độ sâu cây.
- `min_samples_split`: số mẫu tối thiểu để tiếp tục chia.
- `min_samples_leaf`: số mẫu tối thiểu trong một lá.
- **Pruning** (cắt tỉa): xây cây đầy đủ rồi cắt nhánh nào không cải thiện validation.

## 5. Random Forest

Một cây dễ overfit. Ý tưởng Random Forest: train **nhiều cây** trên **dữ liệu hơi khác nhau**, rồi bầu chọn.

Hai nguồn ngẫu nhiên:
1. **Bagging (Bootstrap Aggregation)**: mỗi cây được train trên một tập bootstrap (sample có hoàn lại từ tập gốc, cùng cỡ).
2. **Random subspace**: ở **mỗi nút**, chỉ xét một tập con ngẫu nhiên các feature (thường $\sqrt{d}$ feature trong $d$).

Hai cơ chế này làm các cây ít tương quan với nhau → khi bầu chọn, lỗi triệt tiêu nhau. Đây là lý do Random Forest gần như **luôn** tốt hơn Decision Tree đơn lẻ.

Một bonus đẹp: bootstrap loại ra ~37% mẫu mỗi cây (gọi là **out-of-bag**). Có thể dùng OOB làm validation set miễn phí.

# THỰC HÀNH: Phân loại thuốc với Decision Tree + Random Forest

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

np.random.seed(42)

df = pd.read_csv('data/drug200.csv')
print(df.head())
print(f'\nShape: {df.shape}')
print(f'Drug counts:\n{df["Drug"].value_counts()}')

In [ ]:
# Encode categorical features. BP, Cholesterol có thứ tự (LOW < NORMAL < HIGH)
# nên dùng ordinal encoding hợp lý hơn one-hot.
df_enc = df.copy()
df_enc['Sex']         = df_enc['Sex'].map({'F': 0, 'M': 1})
df_enc['BP']          = df_enc['BP'].map({'LOW': 0, 'NORMAL': 1, 'HIGH': 2})
df_enc['Cholesterol'] = df_enc['Cholesterol'].map({'NORMAL': 0, 'HIGH': 1})

X = df_enc.drop('Drug', axis=1).values
le_y = LabelEncoder()
y = le_y.fit_transform(df_enc['Drug'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
feature_names = df_enc.drop('Drug', axis=1).columns.tolist()
class_names = le_y.classes_.tolist()

print(f'Features: {feature_names}')
print(f'Classes:  {class_names}')
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

### 1. Decision Tree đơn lẻ

In [ ]:
# Train cây không giới hạn — sẽ overfit
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)
print(f'Cây không giới hạn:')
print(f'  Train acc: {dt_full.score(X_train, y_train)*100:.2f}%')
print(f'  Test  acc: {dt_full.score(X_test, y_test)*100:.2f}%')
print(f'  Độ sâu: {dt_full.get_depth()}, số lá: {dt_full.get_n_leaves()}')

In [ ]:
# Sweep max_depth để xem ảnh hưởng đến overfit
depths = list(range(1, 11))
train_acc, test_acc = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_acc.append(m.score(X_train, y_train))
    test_acc.append(m.score(X_test, y_test))

plt.figure(figsize=(8, 4))
plt.plot(depths, [a*100 for a in train_acc], 'o-', label='Train')
plt.plot(depths, [a*100 for a in test_acc],  's-', label='Test')
plt.xlabel('max_depth'); plt.ylabel('Accuracy (%)')
plt.title('Train vs test accuracy theo max_depth')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

best_d = depths[int(np.argmax(test_acc))]
print(f'max_depth tốt nhất: {best_d}, test acc = {max(test_acc)*100:.2f}%')

In [ ]:
# Vẽ cây với max_depth=4 cho dễ nhìn
dt_small = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_small.fit(X_train, y_train)

plt.figure(figsize=(16, 8))
plot_tree(dt_small, feature_names=feature_names, class_names=class_names,
          filled=True, rounded=True, fontsize=9)
plt.title(f'Decision Tree (max_depth=4)  test_acc = {dt_small.score(X_test, y_test)*100:.2f}%')
plt.show()

### Feature importance

Mỗi feature được Decision Tree gán một mức "quan trọng" — bằng tổng giảm impurity ở các nút nó tham gia chia.

In [ ]:
imp = pd.Series(dt_small.feature_importances_, index=feature_names).sort_values()
imp.plot.barh(figsize=(7, 3))
plt.xlabel('Importance'); plt.title('Feature importance — Decision Tree')
plt.tight_layout(); plt.show()

### 2. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=5,
                            oob_score=True, random_state=42)
rf.fit(X_train, y_train)

print(f'Random Forest (100 cây, max_depth=5):')
print(f'  Train acc: {rf.score(X_train, y_train)*100:.2f}%')
print(f'  Test  acc: {rf.score(X_test, y_test)*100:.2f}%')
print(f'  OOB acc:   {rf.oob_score_*100:.2f}%   (validation miễn phí từ bootstrap)')

In [ ]:
# So sánh DT vs RF khi sweep n_estimators
ns = [1, 5, 10, 25, 50, 100, 200]
rf_scores = []
for n in ns:
    m = RandomForestClassifier(n_estimators=n, max_depth=5, random_state=42)
    m.fit(X_train, y_train)
    rf_scores.append(m.score(X_test, y_test))

plt.figure(figsize=(8, 4))
plt.plot(ns, [s*100 for s in rf_scores], 'o-', label='Random Forest')
plt.axhline(dt_small.score(X_test, y_test) * 100, color='red',
            linestyle='--', label='Decision Tree (max_depth=4)')
plt.xlabel('Số cây'); plt.ylabel('Test accuracy (%)')
plt.title('Càng nhiều cây thì RF càng ổn — đến một điểm bão hoà')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Confusion matrix cho RF
y_pred = rf.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Greens')
ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names); ax.set_yticklabels(class_names)
ax.set_xlabel('Dự đoán'); ax.set_ylabel('Thật')
ax.set_title('Confusion matrix — Random Forest')
for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im); plt.tight_layout(); plt.show()

print(classification_report(y_test, y_pred, target_names=class_names))

## Tổng kết

1. **Decision Tree**: dễ giải thích, nhưng dễ overfit nếu không giới hạn depth.
2. **Random Forest**: trung bình hoá nhiều cây → ổn định, ít overfit, gần như luôn tốt hơn DT đơn.
3. **OOB score** là validation "miễn phí" của RF (không cần chia thêm tập val).
4. Cả hai cho `feature_importance_` — công cụ tốt để hiểu dữ liệu.
5. **Tránh data leakage**: nếu dùng `Pipeline` với scaler, vẫn nhớ split TRƯỚC khi `fit`.

# BÀI TẬP VỀ NHÀ

## Bài 1: Gini vs Entropy
Train 2 Decision Tree với `criterion='gini'` và `criterion='entropy'`, các tham số khác giữ nguyên. So sánh test accuracy. Sự khác biệt có lớn không?

## Bài 2: GridSearchCV cho RF
Dùng `GridSearchCV` để tìm best hyperparams cho Random Forest:
- `n_estimators`: [50, 100, 200]
- `max_depth`: [3, 5, 7, None]
- `min_samples_leaf`: [1, 3, 5]

Báo cáo `best_params_` và best CV score. So sánh với RF mặc định.

*Gợi ý:* `from sklearn.model_selection import GridSearchCV; gs = GridSearchCV(rf, params, cv=5).fit(X_train, y_train)`.

## Bài 3: Tự cài Information Gain
Viết hàm `info_gain(y_parent, y_left, y_right)`:
1. Tính entropy của `y_parent`.
2. Tính trung bình có trọng số entropy của hai con.
3. Trả về phép trừ.

Test: với `y_parent = [0,0,0,1,1,1,1,1]`, `y_left = [0,0,0]`, `y_right = [1,1,1,1,1]` → IG phải bằng entropy của parent (vì hai con đều thuần) ≈ 0.954.

## Bài 4: Vẽ decision boundary 2D
Lấy 2 feature `Age` và `Na_to_K` của drug200. Train DT (max_depth=3) và RF (n_estimators=50, max_depth=3). Vẽ decision boundary 2D bằng `contourf`. So sánh: RF có boundary mượt hơn không?

## Bài 5: Feature importance — RF vs DT
Train cả DT (max_depth=5) và RF (n_estimators=100, max_depth=5). In `feature_importances_` của cả hai, vẽ bar chart so sánh. RF có ổn định hơn DT khi đổi `random_state` không? (chạy 5 lần với 5 seed khác nhau, đo std).

*Gợi ý:* lặp với `random_state in [0,1,2,3,4]`, lưu importance, so sánh `np.std`.